In [1]:
import pandas as pd
import  numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML practical/Processed_data.csv')

In [3]:
data=data.drop(columns='Unnamed: 0')

In [6]:
data.head()

,Severity,Distance(mi),Temperature(F),Wind_Chill(F),Humidity(%),Visibility(mi),Wind_Direction,Wind_Speed(mph),Weather_Condition,Crossing,...,pca_90,pca_91,pca_92,pca_93,pca_94,pca_95,pca_96,pca_97,pca_98,pca_99
0,3,0.000,72.0,72.000000,26.0,10.0,6,10.0,16,0,...,0.011294,-0.023450,0.009454,-0.007929,0.000032,-0.007039,-0.011485,0.015448,-0.012006,-0.003866
1,4,1.002,91.8,54.472104,48.0,10.0,22,18.4,6,0,...,0.039491,0.021280,-0.005106,-0.015760,0.016425,0.039872,-0.047719,-0.000016,-0.013681,-0.010934
2,1,0.000,91.0,91.000000,38.0,10.0,20,7.0,16,0,...,0.012038,0.099881,0.019001,0.063796,0.002496,0.006196,0.051213,-0.014346,-0.050256,-0.007663
3,4,1.500,77.0,77.000000,40.0,10.0,10,12.0,91,0,...,0.008458,-0.063279,-0.003959,-0.030436,-0.020446,-0.017096,-0.009632,0.055119,0.063940,0.017744
4,4,0.180,73.0,54.472104,26.0,10.0,10,15.0,91,0,...,-0.047187,0.015230,0.064401,0.005247,-0.016908,-0.042391,0.024430,-0.047003,0.015520,0.032127


In [4]:
data.shape

(256574, 127)

In [5]:
data.size

32584898

Set Feature and Target columns

In [13]:
X=data.drop(columns='Severity')
y=data['Severity']

Import required libraries

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score , classification_report , confusion_matrix
from sklearn.metrics import roc_auc_score , roc_curve
from sklearn.preprocessing import label_binarize

In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

# AdaBoost Classifier

In [16]:
adb = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # weak learners
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

In [17]:
adb.fit(X_train,y_train)

KeyboardInterrupt: 

In [ ]:
adb_pred=adb.predict(X_test)
train_adb_pred=adb.predict(X_train)

In [ ]:
print("Accuracy Score onn Test data: ",accuracy_score(y_test,adb_pred))
print("Accuracy Score on Train data: "accuracy_score(y_train,train_adb_pred))

Classification Report

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
adb_pred_prob=adb.predict_proba(X_test)

MACRO & WEIGHTED MULTI-CLASS AUC (OVR)

In [ ]:
macro_auc = roc_auc_score(y_test, adb_pred_prob, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_test, adb_pred_prob, multi_class='ovr', average='weighted')

print("Macro AUC     :", macro_auc)
print("Weighted AUC  :", weighted_auc)

ROC Curve per class

In [ ]:
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)

In [ ]:

plt.figure(figsize=(8,6))
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], adb_pred_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {classes[i]}")

plt.plot([0,1], [0,1], 'k--')
plt.title("ROC Curve - AdaBoost")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

Confusion Matrix

In [ ]:
adb_cm = confusion_matrix(y_test, adb_pred)
plt.figure(figsize=(6,5))
sns.heatmap(adb_cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - AdaBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# HistGradient Boosting Classifier

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    random_state=42
)

In [ ]:
hgb.fit(X_train, y_train)

In [ ]:
hgb_pred = hgb.predict(X_test)
train_hgb_pred = hgb.predict(X_train)

In [ ]:
print("Accuracy Score onn Test data: ",accuracy_score(y_test,hgb_pred))
print("Accuracy Score on Train data: ",accuracy_score(y_train,train_hgb_pred))

Classification Report

In [ ]:
print("\n===== HistGradientBoost Classification Report =====")
print(classification_report(y_test, hgb_pred))

In [ ]:
hgb_pred_prob=hgb.predict_proba(X_test)

Macro & Weighted Multi-Class AUC (OVR)

In [ ]:
macro_auc = roc_auc_score(y_test, hgb_pred_prob, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_test, hgb_pred_prob, multi_class='ovr', average='weighted')

print("Macro AUC     :", macro_auc)
print("Weighted AUC  :", weighted_auc)

ROC CURVE FOR EACH CLASS

In [ ]:

plt.figure(figsize=(8,6))
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], hgb_pred_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {classes[i]}")

plt.plot([0,1], [0,1], 'k--')
plt.title("ROC Curve - AdaBoost")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

CONFUSION MATRIX

In [ ]:
hgb_cm = confusion_matrix(y_test, hgb_pred)
plt.figure(figsize=(6,5))
sns.heatmap(hgb_cm, annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix - HistGradientBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# LGBM Classifier

In [ ]:
lgb = LGBMClassifier(
    learning_rate=0.05,
    n_estimators=300,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multiclass',
    num_class=4,
    random_state=42
)

In [ ]:
lgb.fit(X_train, y_train)

In [ ]:
lgb_pred = lgb.predict(X_test)
train_lgb_pred = lgb.predict(X_train)
lgb_pred_prob = lgb.predict_proba(X_test)

In [ ]:
print("Accuracy Score onn Test data: ",accuracy_score(y_test,lgb_pred))
print("Accuracy Score on Train data: ",accuracy_score(y_train,train_lgb_pred))

In [ ]:
print("\n===== LightGBM Classification Report =====")
print(classification_report(y_test, lgb_pred))

Macro & Weighted Multi-Class AUC (OVR)

In [ ]:
macro_auc = roc_auc_score(y_test, lgb_pred_prob, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_test, lgb_pred_prob, multi_class='ovr', average='weighted')

print("Macro AUC     :", macro_auc)
print("Weighted AUC  :", weighted_auc)

ROC Curve per class

In [ ]:
plt.figure(figsize=(8,6))
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], lgb_pred_prob[:, i])
    plt.plot(fpr, tpr, label=f"Class {classes[i]}")

plt.plot([0,1], [0,1], 'k--')
plt.title("ROC Curve - LightGBM")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title("Confusion Matrix - LightGBM")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()